# 🔴 Red-Team Zafiyet Testi: Bozuk Veri Enjeksiyonu & Arrow Parser Exploit (MIME/Null-Byte Attack)
**Hedef Kütüphane:** Hugging Face `datasets` & `pyarrow`  
**Test Eden Rol:** QA & Adli Bilişim Uzmanı  
**Tarih:** 2026-Güz  

---

### 🎯 Testin Amacı ve Tehdit Modeli:
Hugging Face `datasets`, verileri C++ tabanlı **Apache Arrow** tablolarında saklar.
Bu testte:
1. CSV/JSON/Parquet kaynaklarına yerleştirilen kontrolsüz Unicode karakterleri, Null Byte'lar (`\x00`) ve tip uyuşmazlıkları (Type Mismatch),
2. `dataset.map()` fonksiyonunda Python ile Arrow C++ çekirdeği arasındaki tip dönüşüm hataları ve pipeline kilitlenmesi,
3. Veri güvenliği savunma yöntemleri test edilecektir.


In [5]:
import tempfile
import json
import os
from datasets import load_dataset, Dataset

print("[*] Bozuk veri enjeksiyon testi hazirlaniyor...")


[*] Bozuk veri enjeksiyon testi hazirlaniyor...


## 1. Zafiyet Senaryosu: Malformed JSON ve Schema Poisoning
Saldırgan aynı sütunda bazen tamsayı (`int`), bazen karmaşık nesne (`dict`), bazen de `null-byte` barındıran kirli bir JSON dosyası sunar.


In [6]:
# Bozuk JSON dosyasi uretimi
corrupted_records = [
    {"id": 1, "data": "Normal metin verisi", "metadata": "v1"},
    {"id": 2, "data": "Ikinci normal kayit", "metadata": "v2"},
    # 💥 SALDIRI SATIRI: Tip uyusmazligi ve Arrow Schema Exploit
    {"id": "UC_STR_ID_999", "data": "Zararli Unicode: \x00\x01\x02\x03", "metadata": {"exploit": True, "nested": [1, 2, 3]}},
    {"id": 4, "data": None, "metadata": 999999999}
]

with tempfile.NamedTemporaryFile('w', suffix='.json', delete=False, encoding='utf-8') as f:
    for rec in corrupted_records:
        f.write(json.dumps(rec) + "\n")
    temp_json_path = f.name

print(f"[*] Zararli JSON olusturuldu: {temp_json_path}")


[*] Zararli JSON olusturuldu: C:\Users\ASUS\AppData\Local\Temp\tmplak_1eq1.json


## 2. Arrow Parser'ın Hata Vermesi & Kırılma Noktası


In [7]:
try:
    print("[*] load_dataset ile bozuk JSON yuklenmeye calisiliyor...")
    # Tip cikarimi sirasinda Arrow C++ seviyesinde hata yakalama
    ds_broken = load_dataset('json', data_files=temp_json_path, split='train')
    print("[!] Yukleme basarili: ", ds_broken)
    
    # Map islemi sirasinda tip uyusmazliginin tetiklenmesi
    def risky_map(x):
        return {"processed_len": len(x["data"]) if x["data"] else 0}
        
    ds_broken = ds_broken.map(risky_map)
    print("[!] Map sonucu: ", ds_broken)
except Exception as e:
    print(f"\n[💥 ADLI TESPIT - ARROW/DATASET COKTU]:")
    print(f"Hata Turu: {type(e).__name__}")
    print(f"Hata Mesaji: {e}")
finally:
    if os.path.exists(temp_json_path):
        os.remove(temp_json_path)


[*] load_dataset ile bozuk JSON yuklenmeye calisiliyor...


Generating train split: 0 examples [00:00, ? examples/s]

[!] Yukleme basarili:  Dataset({
    features: ['id', 'data', 'metadata'],
    num_rows: 4
})


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

[!] Map sonucu:  Dataset({
    features: ['id', 'data', 'metadata', 'processed_len'],
    num_rows: 4
})


## 📌 Sonuç ve Adli Değerlendirme (Red-Team Değerlendirme Notları)
1. **Bulgu:** Hugging Face `datasets`, şema çıkarımını (schema inference) ilk birkaç satıra bakarak yapar. Eğer saldırgan bozuk veya iç içe veri tiplerini daha sonraki satırlara gizlerse, `map()` veya PyTorch `DataLoader` aşamasında tüm eğitim hattı çöker.
2. **Mühendislik Çözümü:** `datasets.Features` sınıfı kullanılarak katı (strict) şema zorunlu tutulmalı, dinamik tip çıkarımına güvenilmemelidir.
